# Reading, PA â€” Land Value Tax Model

**Reform:** 4:1 split-rate (land taxed at 4Ã— the improvement rate), revenue-neutral  
**Levy scope:** City of Reading municipal real estate tax only  
**Assessment base year:** 1994 (Berks County last county-wide reassessment)  
**Tax rate source:** City of Reading 2026 Tax Rates, adopted Dec 12, 2025  
**Data source:** Berks County GIS / CAMA_Master, downloaded via `berks_open_avmkit`

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `assr_land_value` | CAMA LAND_VALUE, 1994 base year |
| Improvement value | `assr_impr_value` | CAMA BLDG_VALUE, 1994 base year |
| Total assessed value | `assr_market_value` | = land + impr for 99.4% of parcels |
| Property class | `category_code` | R / A / C / I / F / E / UE / UT |
| Land use code | `luc` | PA STEB-style; R: 10xâ€“16x; C/I: 4xxx |
| Municipality | `neighborhood` | `'READING'` filters to city parcels |
| Parcel ID | `key` | Berks County PROPID (unique) |
| Vacant flag | `is_vacant` | True when SFLA=0 and bldg_value=0 |

**Assessment ratio:** 100% â€” millage is applied directly to 1994 assessed values  
**Millage source:** City of Reading 2026 Tax Rates PDF (readingpa.gov)  
**Exemptions:** CLASS=E parcels are fully exempt (assr_market_value=0 in assessor records)

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'reading'
STATE_FIPS = '42'
COUNTY_FIPS = '011'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

# 2026 adopted city real estate tax: 18.717 GF + 0.30 Shade Tree + 0.20 Library
# Source: City of Reading 2026 Tax Rates PDF (readingpa.gov), adopted Dec 12, 2025
CITY_MILLAGE = 19.217

# Revenue validation anchor:
# 2024 actual total property tax revenue: $25,693,646 at 18.129 mills
# (City of Reading COW Budget Review, Nov 17, 2025)
# Scaled to 2026 adopted rate: $25,693,646 Ã— (19.217 / 18.129) = ~$27,235,265
OFFICIAL_REVENUE = 27_235_265

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 â€” Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
AVM_PARCEL_PATH = Path('C:/projects/berks_open_avmkit/notebooks/pipeline/data/us-pa-berks/in/berks_parcels.parquet')

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    if not AVM_PARCEL_PATH.exists():
        raise FileNotFoundError(
            f"Source parcel data not found at {AVM_PARCEL_PATH}\n"
            "Run download_berks_parcels.py in berks_open_avmkit first."
        )
    gdf_all = gpd.read_parquet(AVM_PARCEL_PATH)
    gdf = gdf_all[gdf_all['neighborhood'] == 'READING'].copy()
    gdf.reset_index(drop=True, inplace=True)
    gdf.to_parquet(PARCEL_PATH)
    print(f"Filtered and cached {len(gdf):,} Reading parcels")

for col in ['assr_land_value', 'assr_impr_value', 'assr_market_value']:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

print(f"\nTotal parcels: {len(gdf):,}")
print(f"\ncategory_code distribution:")
print(gdf['category_code'].value_counts())
print(f"\nValue column stats:")
print(gdf[['assr_land_value','assr_impr_value','assr_market_value']].describe().round(0))
print(f"\nParcels with assr_land_value=0:  {(gdf['assr_land_value']==0).sum():,}")
print(f"Parcels with assr_impr_value=0:  {(gdf['assr_impr_value']==0).sum():,}")
print(f"Parcels with assr_market_value=0: {(gdf['assr_market_value']==0).sum():,}")

Filtered and cached 26,299 Reading parcels

Total parcels: 26,299

category_code distribution:
category_code
R     22613
C      2505
E       861
I       210
A        50
UE       35
UT       24
F         1
Name: count, dtype: int64

Value column stats:
       assr_land_value  assr_impr_value  assr_market_value
count          26299.0          26299.0            26299.0
mean           19645.0          59255.0            53479.0
std            73987.0         624819.0           181366.0
min                0.0              0.0                0.0
25%             7700.0          16500.0            25000.0
50%            12000.0          24400.0            36000.0
75%            16100.0          36700.0            50500.0
max          4118200.0       62372500.0         13728000.0

Parcels with assr_land_value=0:  119
Parcels with assr_impr_value=0:  1,571
Parcels with assr_market_value=0: 870


## Section 3 â€” Classify and Validate

In [3]:
# CLASS=E parcels are fully exempt; assessor sets assr_market_value=0 for these
gdf['full_exmp'] = (gdf['category_code'] == 'E').astype(int)

# Property category mapping
# Berks County R-parcel LUC codes follow PA STEB residential classification:
#   Suffix 'R' = rowhouse/attached (112R=2-story, 117R=3-story, 138R=3-story, etc.)
#   162, 163 = 3-story row-type without explicit R suffix
#   12x = small multi-family (121A, 121D: ~2.5 stories, 2-4 units)
#   13x = small multi-family (133, 138: larger 2-3 unit buildings)
#   149, 153 = other residential (misc. types, low improvement value)
#   All others in R = single family detached/semi-detached
# Commercial LUC codes:
#   42xx on C parcels = apartment buildings in commercial zone â†’ Large MF
#   23xx/24xx = smaller multi-unit residential commercial
#   51xx-59xx = industrial/warehouse
#   All others in C = general commercial

def classify_parcel(row):
    cat = str(row.get('category_code', ''))
    luc = str(row.get('luc', ''))

    if cat == 'E':
        return 'Exempt'
    elif cat == 'R':
        if luc.endswith('R') or luc in ['162', '163']:
            return 'Townhome / Rowhouse'
        elif luc[:2] in ['12', '13']:
            return 'Small Multi-Family (2-4 units)'
        elif luc in ['149', '153']:
            return 'Other Residential'
        else:
            return 'Single Family Residential'
    elif cat == 'A':
        return 'Large Multi-Family (5+ units)'
    elif cat in ['C', 'I']:
        luc_prefix = luc[:2]
        if luc_prefix == '42':
            return 'Large Multi-Family (5+ units)'
        elif luc_prefix in ['23', '24']:
            return 'Small Multi-Family (2-4 units)'
        elif cat == 'I' or luc_prefix in ['51', '52', '58', '59']:
            return 'Industrial'
        else:
            return 'Commercial'
    elif cat == 'F':
        return 'Agricultural'
    elif cat in ['UE', 'UT']:
        return 'Other'
    else:
        return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

# Override: zero improvement value â†’ Vacant Land (regardless of use code)
gdf.loc[gdf['assr_impr_value'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nNull categories: {gdf['PROPERTY_CATEGORY'].isna().sum()}")

Property category distribution:
PROPERTY_CATEGORY
Townhome / Rowhouse               15603
Single Family Residential          4053
Vacant Land                        1571
Small Multi-Family (2-4 units)     1571
Commercial                         1426
Large Multi-Family (5+ units)       777
Other Residential                   541
Exempt                              504
Industrial                          236
Other                                16
Agricultural                          1
Name: count, dtype: int64

Null categories: 0


## Section 4 â€” Current Tax Model

Reading uses the standard Pennsylvania system: millage (mills = \$/1,000 assessed value)
applied directly to the county-assessed value (1994 base year). No assessment ratio to apply â€”
the millage rate is calibrated to the 1994 base already.

**Validation note:** The modeled gross levy should be compared to the official 2026 adopted levy.
2024 actual total property tax collections were \$25,693,646 at 18.129 mills. Scaled to the
2026 adopted rate of 19.217 mills (6% increase, Dec 12 2025) â†’ ~\$27,235,265. Note that
actual cash collections run ~9% below the gross levy due to delinquency (consistent with
Reading's history as a low-income city that recently exited Act 47 distress).

In [4]:
gdf['taxable_land_value'] = gdf['assr_land_value'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['assr_impr_value'].clip(lower=0)

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='assr_market_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled gross levy:  ${current_revenue:,.0f}")
print(f"Official target:     ${OFFICIAL_REVENUE:,.0f}  (2024 actual scaled to 2026 rate)")
print(f"Gap:                 {gap_pct:+.2f}%")
print(f"\n(2024 actual cash collections were $25,693,646; ~9% delinquency explains")
print(f" the difference between modeled gross levy and reported collections.)")

Modeled gross levy:  $27,027,087
Official target:     $27,235,265  (2024 actual scaled to 2026 rate)
Gap:                 -0.76%

(2024 actual cash collections were $25,693,646; ~9% delinquency explains
 the difference between modeled gross levy and reported collections.)


## Section 5 â€” 4:1 Split-Rate Model

In [5]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"(Prior flat rate:    {CITY_MILLAGE:.3f} mills)")
print(f"Revenue check:       ${new_revenue:,.0f}")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Reading, PA â€” 4:1 Split-Rate Tax Impact")

Land millage:        40.4355 mills
Improvement millage: 10.1089 mills
(Prior flat rate:    19.217 mills)
Revenue check:       $27,027,087


Reading, PA â€” 4:1 Split-Rate Tax Impact


                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
           Townhome / Rowhouse  15603        $132,920        1.3%         $9          $14    3.3%       2.4%            28.9%            18.9%
     Single Family Residential   4053         $98,280        1.7%        $24          $-5    3.8%      -0.4%            26.7%            17.5%
                   Vacant Land   1571        $181,119       77.4%       $115          $21   80.4%     110.4%            74.2%             2.5%
Small Multi-Family (2-4 units)   1571        $-65,379       -4.4%       $-42          $-7    0.3%      -0.8%            21.3%            33.4%
                    Commercial   1426        $-65,220       -2.1%       $-46          $51   10.5%       6.7%            45.2%            22.0%
 Large Multi-Family (5+ units)    777       $-157,253       -4.2%      $-202         $105   20.7%      12.8%            52.8%            23.9%

## Section 6 â€” Exploration Charts

In [6]:
# Category-level median tax change
fig, ax = plt.subplots(figsize=(10, 6))
cat_med = (
    gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt']
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)
colors = ['#d62728' if v > 0 else '#2ca02c' for v in cat_med.values]
cat_med.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Reading, PA â€” Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("Saved category_preview.png")

Saved category_preview.png


## Section 7 â€” Census Join + Export

In [7]:
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf carries demographics baked in â€” do NOT merge census_data again
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out â€” skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [8]:
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ reading: 26,299 rows → ../../analysis/data/reading.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | -0.76% vs. implied 2026 levy ~$27.24M (2024 actual $25.69M at 18.129 mills, scaled to 2026 adopted rate 19.217 mills) |
| Parcel count | 26,299 Reading parcels (neighborhood == READING in Berks County GIS) |
| Census coverage | 100.0% matched to Berks County block groups |
| PNGs generated | 7 of 7 |
| SFR median change | -0.44% (slight decrease; small magnitude reflects dense urban rowhouse city with high land fractions) |
| Townhome/Rowhouse median change | +2.42% (slight increase; Reading rowhouses in dense urban core carry high land-to-improvement ratios under 1994 base year) |
| Vacant land median change | +110.42% |

**Assessment base year note:** Berks County last reassessed in 1994. All assessed values are 1994-era. Millage rates are calibrated to this base, so the millage × assessed value calculation is internally consistent. The 1994 values do not reflect current market values, but the relative land/improvement splits within the 1994 base are used as a proxy for current splits. CCAO-style AVM land/improvement splits are available in the sibling project  but were not used here; see  for an AVM-based sensitivity analysis.

**Large Multi-Family note:** Median parcel change is +12.77% despite aggregate -4.2%. This bimodal distribution reflects older urban apartment buildings (low 1994 assessed improvements, high land fraction → increases) vs. a few large high-quality complexes (high improvement values → decreases that pull the aggregate negative).